In [34]:
import pandas as pd
import numpy as np
import re
import json


In [35]:
emissions_data_batched = pd.read_csv('emissions_batched_benchmakrs.csv')
emissions_data_nonbatched = pd.read_csv('emissions_benchmakrs.csv')

leaderboard_path = 'arena-hard-auto/leaderboard/arena_hard_leaderboard_20240817.csv'
leaderboard_df = pd.read_csv(leaderboard_path)

In [36]:
# Clean the leaderboard DataFrame
# Extract CI values from the CI column and add them as new columns
leaderboard_df['CI_lower'] = leaderboard_df['CI'].apply(lambda x: float(x.split(',')[0].strip('("(').strip()))
leaderboard_df['CI_upper'] = leaderboard_df['CI'].apply(lambda x: float(x.split(',')[1].strip(')+"').strip()))

# Rename columns to avoid conflicts
leaderboard_df = leaderboard_df.rename(columns={
    'model': 'model_name',
    'avg_tokens': 'output_tok',
    'score': 'arena_score',
    'rating_q025': '95_conf_minus',
    'rating_q975': '95_conf_plus'
})


In [37]:
leaderboard_df

,model_name,arena_score,95_conf_minus,95_conf_plus,CI,output_tok,date,CI_lower,CI_upper
0,gpt-4o,88.09,86.60,89.44,"(-1.49, +1.35)",696.0,2024-08-17,-1.49,1.35
1,gpt-4o-2024-08-06_guided,87.75,86.27,89.18,"(-1.48, +1.43)",625.0,2024-08-17,-1.48,1.43
2,gpt-4o-2024-08-06,86.55,85.08,88.10,"(-1.47, +1.55)",616.0,2024-08-17,-1.47,1.55
3,gpt-4o-mini_guided,86.27,84.67,87.86,"(-1.60, +1.59)",678.0,2024-08-17,-1.60,1.59
4,mistral_large_2_guided,83.63,81.89,85.41,"(-1.74, +1.78)",777.0,2024-08-17,-1.74,1.78
5,mistral_large_2,80.96,78.94,82.60,"(-2.02, +1.64)",684.0,2024-08-17,-2.02,1.64
6,gpt-4o-mini,80.09,78.17,81.92,"(-1.92, +1.83)",631.0,2024-08-17,-1.92,1.83
7,llama3_1_405b_guided,79.47,77.49,81.40,"(-1.98, +1.93)",649.0,2024-08-17,-1.98,1.93
8,llama3_1_70b_guided,78.29,76.17,80.12,"(-2.12, +1.83)",646.0,2024-08-17,-2.12,1.83
9,llama3_1_70b_fp8_guided,75.14,72.89,77.38,"(-2.25, +2.24)",646.0,2024-08-17,-2.25,2.24


In [38]:
input_tok_map = {
    'llama3_1': 135.142,
    'llama3': 135.142,
    'mistral_nemo': 143.19,
}

input_tok_map_guided = {
    'llama3_1': 874.592,
    'llama3': 874.592,
    'mistral_nemo': 921.216,
}

In [39]:
emissions_data_batched.head(5)

,timestamp,project_name,run_id,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,...,cpu_count,cpu_model,gpu_count,gpu_model,longitude,latitude,ram_total_size,tracking_mode,on_cloud,pue
0,2024-08-13T10:18:36,llama3_1_8b-arena-hard-v0.1-4gpus-run_1,a29058ee-62ad-4d69-bfe1-3eb13e10372d,397.887474,0.029054,0.000073,42.5,287.699319,68.162162,0.005731,...,48,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765766,machine,N,1.22
1,2024-08-13T10:25:54,llama3_1_8b-arena-hard-v0.1-4gpus-run_2,a1f46fd3-1e50-42f2-94b5-49016e012a86,424.195516,0.032371,0.000076,42.5,287.760744,68.162162,0.006110,...,48,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765766,machine,N,1.22
2,2024-08-13T10:33:21,llama3_1_8b-arena-hard-v0.1-4gpus-run_3,084245c7-74ca-4706-aae5-3d411eef3567,431.738808,0.033038,0.000077,42.5,285.797297,68.162162,0.006218,...,48,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765766,machine,N,1.22
3,2024-08-13T10:40:26,llama3_1_8b-arena-hard-v0.1-4gpus-run_4,52cca1da-8845-42b0-a78c-923668c3a2ea,409.483063,0.031534,0.000077,42.5,287.372991,68.162162,0.005898,...,48,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765766,machine,N,1.22
4,2024-08-13T10:47:25,llama3_1_8b-arena-hard-v0.1-4gpus-run_5,b3224b99-b7a8-425e-9733-f201dcb9336c,404.363748,0.031162,0.000077,42.5,286.115885,68.162162,0.005824,...,48,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765766,machine,N,1.22


In [40]:
# Strip "-run_x" and "-xgpus" and "-arena-hard-v0.1" from the project_name and create a new column for model_name
emissions_data_batched['model_name'] = emissions_data_batched['project_name'].apply(lambda x: re.sub(r'-run_\d+', '', re.sub(r'-arena-hard-v0.1', '', x)))

In [41]:
emissions_data_batched.head(5)

,timestamp,project_name,run_id,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,...,cpu_model,gpu_count,gpu_model,longitude,latitude,ram_total_size,tracking_mode,on_cloud,pue,model_name
0,2024-08-13T10:18:36,llama3_1_8b-arena-hard-v0.1-4gpus-run_1,a29058ee-62ad-4d69-bfe1-3eb13e10372d,397.887474,0.029054,0.000073,42.5,287.699319,68.162162,0.005731,...,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765766,machine,N,1.22,llama3_1_8b-4gpus
1,2024-08-13T10:25:54,llama3_1_8b-arena-hard-v0.1-4gpus-run_2,a1f46fd3-1e50-42f2-94b5-49016e012a86,424.195516,0.032371,0.000076,42.5,287.760744,68.162162,0.006110,...,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765766,machine,N,1.22,llama3_1_8b-4gpus
2,2024-08-13T10:33:21,llama3_1_8b-arena-hard-v0.1-4gpus-run_3,084245c7-74ca-4706-aae5-3d411eef3567,431.738808,0.033038,0.000077,42.5,285.797297,68.162162,0.006218,...,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765766,machine,N,1.22,llama3_1_8b-4gpus
3,2024-08-13T10:40:26,llama3_1_8b-arena-hard-v0.1-4gpus-run_4,52cca1da-8845-42b0-a78c-923668c3a2ea,409.483063,0.031534,0.000077,42.5,287.372991,68.162162,0.005898,...,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765766,machine,N,1.22,llama3_1_8b-4gpus
4,2024-08-13T10:47:25,llama3_1_8b-arena-hard-v0.1-4gpus-run_5,b3224b99-b7a8-425e-9733-f201dcb9336c,404.363748,0.031162,0.000077,42.5,286.115885,68.162162,0.005824,...,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765766,machine,N,1.22,llama3_1_8b-4gpus


In [42]:
emissions_data_nonbatched['model_name'] = emissions_data_nonbatched['project_name'].apply(lambda x: re.sub(r'-arena-hard-v0.1', '', x))

In [43]:
emissions_data_nonbatched.head(5)

,timestamp,project_name,run_id,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,...,cpu_model,gpu_count,gpu_model,longitude,latitude,ram_total_size,tracking_mode,on_cloud,pue,model_name
0,2024-08-05T11:52:32,llama3_1_70b-arena-hard-v0.1-8gpus,845ed7f7-e6df-420b-b0f6-74c463c5f85a,1867.868346,0.347547,0.000186,42.5,577.583519,273.079635,0.026902,...,AMD EPYC 7R13 Processor,8,8 x NVIDIA L4,-83.0061,39.9625,728.212360,machine,N,1.22,llama3_1_70b-8gpus
1,2024-08-06T08:09:33,llama3_1_70b_awq_int4-arena-hard-v0.1-8gpus,c92fea3c-ce86-4c7e-8731-c7cd7fb5f115,913.961446,0.150479,0.000165,42.5,577.146518,273.079618,0.013164,...,AMD EPYC 7R13 Processor,8,8 x NVIDIA L4,-83.0061,39.9625,728.212315,machine,N,1.22,llama3_1_70b_awq_int4-8gpus
2,2024-08-06T09:04:39,llama3_1_70b_awq_int4-arena-hard-v0.1-8gpus,895031ee-302e-47d7-99c1-e8e383876f8f,823.504677,0.132878,0.000161,42.5,562.538170,273.079618,0.011861,...,AMD EPYC 7R13 Processor,8,8 x NVIDIA L4,-83.0061,39.9625,728.212315,machine,N,1.22,llama3_1_70b_awq_int4-8gpus
3,2024-08-06T12:57:46,llama3_1_70b_fp8-arena-hard-v0.1-4gpus,cbd8317a-68cf-46a3-8eea-adcb3d9d24f3,1466.116555,0.129593,0.000088,42.5,288.045262,68.162164,0.021116,...,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765770,machine,N,1.22,llama3_1_70b_fp8-4gpus
4,2024-08-06T13:42:14,llama3_1_70b_awq_int4-arena-hard-v0.1-4gpus,f3fd9f2b-b4f8-4d49-bf59-d21af0c60226,828.743464,0.070360,0.000085,42.5,289.168718,68.162164,0.011936,...,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765770,machine,N,1.22,llama3_1_70b_awq_int4-4gpus


In [44]:
# Group by the new model_name and calculate the mean for the selected columns
df_batched = emissions_data_batched.groupby('model_name').agg({
    'duration': 'mean',
    'emissions': 'mean',
    'emissions_rate': 'mean',
    'cpu_power': 'mean',
    'gpu_power': 'mean',
    'ram_power': 'mean',
    'cpu_energy': 'mean',
    'gpu_energy': 'mean',
    'ram_energy': 'mean',
    'energy_consumed': 'mean',
    'gpu_count': 'first'}).reset_index()
    
df_nonbatched = emissions_data_nonbatched.groupby('model_name').agg({
    'duration': 'mean',
    'emissions': 'mean',
    'emissions_rate': 'mean',
    'cpu_power': 'mean',
    'gpu_power': 'mean',
    'ram_power': 'mean',
    'cpu_energy': 'mean',
    'gpu_energy': 'mean',
    'ram_energy': 'mean',
    'energy_consumed': 'mean',
    'gpu_count': 'first'}).reset_index()

In [45]:
df_batched.head(5)

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,energy_consumed,gpu_count
0,llama3_1_70b_awq_int4-4gpus,1480.617988,0.132385,0.000089,42.5,288.748807,68.162162,0.021325,0.143604,0.034161,0.199090,4
1,llama3_1_70b_awq_int4_guided-4gpus,3488.539857,0.214276,0.000069,42.5,200.229282,68.162162,0.050245,0.191476,0.080524,0.322245,4
2,llama3_1_70b_fp8-4gpus,2002.761728,0.179008,0.000089,42.5,288.561002,68.162162,0.028845,0.194161,0.046200,0.269207,4
3,llama3_1_70b_fp8-8gpus,1220.240308,0.211623,0.000173,42.5,581.537797,273.079614,0.017575,0.187966,0.112714,0.318255,8
4,llama3_1_70b_fp8_guided-4gpus,2879.996033,0.251599,0.000087,42.5,288.047993,68.162162,0.041480,0.270455,0.066439,0.378373,4


In [46]:
df_nonbatched.head(5)

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,energy_consumed,gpu_count
0,llama3_1_70b-8gpus,2513.262675,0.471343,0.000187,42.5,563.751591,273.079628,0.036198,0.440552,0.232091,0.708841,8
1,llama3_1_70b_awq_int4-4gpus,828.743464,0.070360,0.000085,42.5,289.168718,68.162164,0.011936,0.074753,0.019123,0.105812,4
2,llama3_1_70b_awq_int4-8gpus,890.192632,0.146427,0.000164,42.5,573.461085,273.079615,0.012821,0.125151,0.082235,0.220208,8
3,llama3_1_70b_awq_int4-8gpus_guided,1692.569733,0.281111,0.000166,42.5,575.850005,273.079634,0.024378,0.242026,0.156352,0.422756,8
4,llama3_1_70b_fp8-4gpus,1466.116555,0.129593,0.000088,42.5,288.045262,68.162164,0.021116,0.139951,0.033825,0.194892,4


In [47]:
df = pd.concat([df_batched, df_nonbatched], ignore_index=True)
df = df.groupby('model_name').agg({
    'duration': 'mean',
    'emissions': 'mean',
    'emissions_rate': 'mean',
    'cpu_power': 'mean',
    'gpu_power': 'mean',
    'ram_power': 'mean',
    'cpu_energy': 'mean',
    'gpu_energy': 'mean',
    'ram_energy': 'mean',
    'energy_consumed': 'mean',
    'gpu_count': 'first'}).reset_index()
df.head(10)

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,energy_consumed,gpu_count
0,llama3_1_70b-8gpus,2513.262675,0.471343,0.000187,42.5,563.751591,273.079628,0.036198,0.440552,0.232091,0.708841,8
1,llama3_1_70b_awq_int4-4gpus,1154.680726,0.101372,0.000087,42.5,288.958763,68.162163,0.016631,0.109179,0.026642,0.152451,4
2,llama3_1_70b_awq_int4-8gpus,890.192632,0.146427,0.000164,42.5,573.461085,273.079615,0.012821,0.125151,0.082235,0.220208,8
3,llama3_1_70b_awq_int4-8gpus_guided,1692.569733,0.281111,0.000166,42.5,575.850005,273.079634,0.024378,0.242026,0.156352,0.422756,8
4,llama3_1_70b_awq_int4_guided-4gpus,3488.539857,0.214276,0.000069,42.5,200.229282,68.162162,0.050245,0.191476,0.080524,0.322245,4
5,llama3_1_70b_fp8-4gpus,1734.439142,0.154301,0.000089,42.5,288.303132,68.162163,0.024981,0.167056,0.040013,0.232050,4
6,llama3_1_70b_fp8-8gpus,1220.240308,0.211623,0.000173,42.5,581.537797,273.079614,0.017575,0.187966,0.112714,0.318255,8
7,llama3_1_70b_fp8_guided-4gpus,2879.996033,0.251599,0.000087,42.5,288.047993,68.162162,0.041480,0.270455,0.066439,0.378373,4
8,llama3_1_70b_fp8_guided-8gpus,1882.436985,0.321359,0.000171,42.5,579.537203,273.079614,0.027112,0.282304,0.173868,0.483285,8
9,llama3_1_8b-4gpus,406.711346,0.031154,0.000077,42.5,290.192641,68.162162,0.005858,0.031608,0.009385,0.046852,4


In [48]:
df['model_id'] = df['model_name']
df['model_name'] = df['model_name'].apply(lambda x: re.sub(r'-\d+gpus', '', x))

In [49]:
df

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,energy_consumed,gpu_count,model_id
0,llama3_1_70b,2513.262675,0.471343,0.000187,42.5,563.751591,273.079628,0.036198,0.440552,0.232091,0.708841,8,llama3_1_70b-8gpus
1,llama3_1_70b_awq_int4,1154.680726,0.101372,0.000087,42.5,288.958763,68.162163,0.016631,0.109179,0.026642,0.152451,4,llama3_1_70b_awq_int4-4gpus
2,llama3_1_70b_awq_int4,890.192632,0.146427,0.000164,42.5,573.461085,273.079615,0.012821,0.125151,0.082235,0.220208,8,llama3_1_70b_awq_int4-8gpus
3,llama3_1_70b_awq_int4_guided,1692.569733,0.281111,0.000166,42.5,575.850005,273.079634,0.024378,0.242026,0.156352,0.422756,8,llama3_1_70b_awq_int4-8gpus_guided
4,llama3_1_70b_awq_int4_guided,3488.539857,0.214276,0.000069,42.5,200.229282,68.162162,0.050245,0.191476,0.080524,0.322245,4,llama3_1_70b_awq_int4_guided-4gpus
5,llama3_1_70b_fp8,1734.439142,0.154301,0.000089,42.5,288.303132,68.162163,0.024981,0.167056,0.040013,0.232050,4,llama3_1_70b_fp8-4gpus
6,llama3_1_70b_fp8,1220.240308,0.211623,0.000173,42.5,581.537797,273.079614,0.017575,0.187966,0.112714,0.318255,8,llama3_1_70b_fp8-8gpus
7,llama3_1_70b_fp8_guided,2879.996033,0.251599,0.000087,42.5,288.047993,68.162162,0.041480,0.270455,0.066439,0.378373,4,llama3_1_70b_fp8_guided-4gpus
8,llama3_1_70b_fp8_guided,1882.436985,0.321359,0.000171,42.5,579.537203,273.079614,0.027112,0.282304,0.173868,0.483285,8,llama3_1_70b_fp8_guided-8gpus
9,llama3_1_8b,406.711346,0.031154,0.000077,42.5,290.192641,68.162162,0.005858,0.031608,0.009385,0.046852,4,llama3_1_8b-4gpus


In [50]:
# Perform an outer join to include all models from both DataFrames
df = pd.merge(df, leaderboard_df, on='model_name', how='outer')

# Extract models that have no energy data (all columns from the original df will be NaN)
models_with_no_energy_data = df[df['duration'].isna()]['model_name'].unique()

# Extract models that have no arena score data (all columns from the leaderboard_df will be NaN)
models_with_no_arena_score = df[df['arena_score'].isna()]['model_name'].unique()

df['model_id'] = df['model_id'].fillna(df['model_name'])

df.head(10)

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,...,gpu_count,model_id,arena_score,95_conf_minus,95_conf_plus,CI,output_tok,date,CI_lower,CI_upper
0,gpt-4-0314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,gpt-4-0314,50.00,50.00,50.00,"(-0.00, +0.00)",423.0,2024-08-17,-0.00,0.00
1,gpt-4o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,gpt-4o,88.09,86.60,89.44,"(-1.49, +1.35)",696.0,2024-08-17,-1.49,1.35
2,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,gpt-4o-2024-08-06,86.55,85.08,88.10,"(-1.47, +1.55)",616.0,2024-08-17,-1.47,1.55
3,gpt-4o-2024-08-06_guided,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,gpt-4o-2024-08-06_guided,87.75,86.27,89.18,"(-1.48, +1.43)",625.0,2024-08-17,-1.48,1.43
4,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,gpt-4o-mini,80.09,78.17,81.92,"(-1.92, +1.83)",631.0,2024-08-17,-1.92,1.83
5,gpt-4o-mini_guided,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,gpt-4o-mini_guided,86.27,84.67,87.86,"(-1.60, +1.59)",678.0,2024-08-17,-1.60,1.59
6,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,llama3_1_405b,72.20,69.91,73.99,"(-2.29, +1.79)",662.0,2024-08-17,-2.29,1.79
7,llama3_1_405b_guided,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,llama3_1_405b_guided,79.47,77.49,81.40,"(-1.98, +1.93)",649.0,2024-08-17,-1.98,1.93
8,llama3_1_70b,2513.262675,0.471343,0.000187,42.5,563.751591,273.079628,0.036198,0.440552,0.232091,...,8.0,llama3_1_70b-8gpus,68.05,65.91,70.15,"(-2.14, +2.10)",681.0,2024-08-17,-2.14,2.10
9,llama3_1_70b_awq_int4,1154.680726,0.101372,0.000087,42.5,288.958763,68.162163,0.016631,0.109179,0.026642,...,4.0,llama3_1_70b_awq_int4-4gpus,63.82,61.24,66.03,"(-2.58, +2.21)",641.0,2024-08-17,-2.58,2.21


In [51]:
no_energy = pd.DataFrame({"no energy data": models_with_no_energy_data})
no_energy

,no energy data
0,gpt-4-0314
1,gpt-4o
2,gpt-4o-2024-08-06
3,gpt-4o-2024-08-06_guided
4,gpt-4o-mini
5,gpt-4o-mini_guided
6,llama3_1_405b
7,llama3_1_405b_guided
8,llama3_1_70b_guided
9,mistral_large_2


In [52]:
no_arena = pd.DataFrame({"no arena score": models_with_no_arena_score})
no_arena

,no arena score
0,llama3_1_8b_fp8
1,llama3_70b
2,llama3_70b_awq_int4
3,llama3_8b
4,llama3_8b_guided


In [53]:
# Create the guided column
df['guided'] = df['model_name'].apply(lambda x: 1 if '_guided' in x else 0)

# Strip "_guided" from the model name
df['model_name'] = df['model_name'].apply(lambda x: re.sub(r'_guided', '', x))

In [54]:
df.head(5)

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,...,model_id,arena_score,95_conf_minus,95_conf_plus,CI,output_tok,date,CI_lower,CI_upper,guided
0,gpt-4-0314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,gpt-4-0314,50.00,50.00,50.00,"(-0.00, +0.00)",423.0,2024-08-17,-0.00,0.00,0
1,gpt-4o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,gpt-4o,88.09,86.60,89.44,"(-1.49, +1.35)",696.0,2024-08-17,-1.49,1.35,0
2,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,gpt-4o-2024-08-06,86.55,85.08,88.10,"(-1.47, +1.55)",616.0,2024-08-17,-1.47,1.55,0
3,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,gpt-4o-2024-08-06_guided,87.75,86.27,89.18,"(-1.48, +1.43)",625.0,2024-08-17,-1.48,1.43,1
4,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,gpt-4o-mini,80.09,78.17,81.92,"(-1.92, +1.83)",631.0,2024-08-17,-1.92,1.83,0


In [55]:
# Add a new column 'quantization' based on the suffixes in the model name
def assign_quantization(model_name):
    if '_fp8' in model_name:
        return 'fp8'
    elif '_awq_int4' in model_name:
        return 'awq_int4'
    else:
        return 'bf16'

In [56]:
df['quantization'] = df['model_name'].apply(assign_quantization)

# Strip the quantization suffixes from the model name
df['model_name'] = df['model_name'].apply(lambda x: re.sub(r'_fp8|_awq_int4', '', x))

In [57]:
df.head(10)

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,...,arena_score,95_conf_minus,95_conf_plus,CI,output_tok,date,CI_lower,CI_upper,guided,quantization
0,gpt-4-0314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,50.00,50.00,50.00,"(-0.00, +0.00)",423.0,2024-08-17,-0.00,0.00,0,bf16
1,gpt-4o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,88.09,86.60,89.44,"(-1.49, +1.35)",696.0,2024-08-17,-1.49,1.35,0,bf16
2,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,86.55,85.08,88.10,"(-1.47, +1.55)",616.0,2024-08-17,-1.47,1.55,0,bf16
3,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,87.75,86.27,89.18,"(-1.48, +1.43)",625.0,2024-08-17,-1.48,1.43,1,bf16
4,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,80.09,78.17,81.92,"(-1.92, +1.83)",631.0,2024-08-17,-1.92,1.83,0,bf16
5,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,86.27,84.67,87.86,"(-1.60, +1.59)",678.0,2024-08-17,-1.60,1.59,1,bf16
6,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,72.20,69.91,73.99,"(-2.29, +1.79)",662.0,2024-08-17,-2.29,1.79,0,bf16
7,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,79.47,77.49,81.40,"(-1.98, +1.93)",649.0,2024-08-17,-1.98,1.93,1,bf16
8,llama3_1_70b,2513.262675,0.471343,0.000187,42.5,563.751591,273.079628,0.036198,0.440552,0.232091,...,68.05,65.91,70.15,"(-2.14, +2.10)",681.0,2024-08-17,-2.14,2.10,0,bf16
9,llama3_1_70b,1154.680726,0.101372,0.000087,42.5,288.958763,68.162163,0.016631,0.109179,0.026642,...,63.82,61.24,66.03,"(-2.58, +2.21)",641.0,2024-08-17,-2.58,2.21,0,awq_int4


In [58]:
# Function to extract model class and remove the suffix from the model name
def extract_param_size(model_name):
    match = re.search(r'_\d+b', model_name)
    if match:
        param_size = match.group(0).lstrip('_')
        param_size = param_size.rstrip('b')
        return int(param_size)
    elif model_name == 'mistral_nemo':
        return 12
    elif model_name == 'mistral_large_2':
        return 123
    return np.nan

In [59]:
df['param_size'] = df['model_name'].apply(extract_param_size)
df['model_class'] = df['model_name'].apply(lambda x: re.sub(r'_\d+b', '', x))

df

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,...,95_conf_plus,CI,output_tok,date,CI_lower,CI_upper,guided,quantization,param_size,model_class
0,gpt-4-0314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,50.00,"(-0.00, +0.00)",423.0,2024-08-17,-0.00,0.00,0,bf16,NaN,gpt-4-0314
1,gpt-4o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,89.44,"(-1.49, +1.35)",696.0,2024-08-17,-1.49,1.35,0,bf16,NaN,gpt-4o
2,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,88.10,"(-1.47, +1.55)",616.0,2024-08-17,-1.47,1.55,0,bf16,NaN,gpt-4o-2024-08-06
3,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,89.18,"(-1.48, +1.43)",625.0,2024-08-17,-1.48,1.43,1,bf16,NaN,gpt-4o-2024-08-06
4,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,81.92,"(-1.92, +1.83)",631.0,2024-08-17,-1.92,1.83,0,bf16,NaN,gpt-4o-mini
5,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,87.86,"(-1.60, +1.59)",678.0,2024-08-17,-1.60,1.59,1,bf16,NaN,gpt-4o-mini
6,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,73.99,"(-2.29, +1.79)",662.0,2024-08-17,-2.29,1.79,0,bf16,405.0,llama3_1
7,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,81.40,"(-1.98, +1.93)",649.0,2024-08-17,-1.98,1.93,1,bf16,405.0,llama3_1
8,llama3_1_70b,2513.262675,0.471343,0.000187,42.5,563.751591,273.079628,0.036198,0.440552,0.232091,...,70.15,"(-2.14, +2.10)",681.0,2024-08-17,-2.14,2.10,0,bf16,70.0,llama3_1
9,llama3_1_70b,1154.680726,0.101372,0.000087,42.5,288.958763,68.162163,0.016631,0.109179,0.026642,...,66.03,"(-2.58, +2.21)",641.0,2024-08-17,-2.58,2.21,0,awq_int4,70.0,llama3_1


In [60]:
def map_input_tok(row):
    if row['guided'] == 1:
        return input_tok_map_guided.get(row['model_class'], 0)
    else:
        return input_tok_map.get(row['model_class'], 0)

In [61]:
df['input_tok'] = df.apply(map_input_tok, axis=1)

In [62]:
df['duration_minutes'] = df['duration'] / 60
df

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,...,output_tok,date,CI_lower,CI_upper,guided,quantization,param_size,model_class,input_tok,duration_minutes
0,gpt-4-0314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,423.0,2024-08-17,-0.00,0.00,0,bf16,NaN,gpt-4-0314,0.000,NaN
1,gpt-4o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,696.0,2024-08-17,-1.49,1.35,0,bf16,NaN,gpt-4o,0.000,NaN
2,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,616.0,2024-08-17,-1.47,1.55,0,bf16,NaN,gpt-4o-2024-08-06,0.000,NaN
3,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,625.0,2024-08-17,-1.48,1.43,1,bf16,NaN,gpt-4o-2024-08-06,0.000,NaN
4,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,631.0,2024-08-17,-1.92,1.83,0,bf16,NaN,gpt-4o-mini,0.000,NaN
5,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,678.0,2024-08-17,-1.60,1.59,1,bf16,NaN,gpt-4o-mini,0.000,NaN
6,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,662.0,2024-08-17,-2.29,1.79,0,bf16,405.0,llama3_1,135.142,NaN
7,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,649.0,2024-08-17,-1.98,1.93,1,bf16,405.0,llama3_1,874.592,NaN
8,llama3_1_70b,2513.262675,0.471343,0.000187,42.5,563.751591,273.079628,0.036198,0.440552,0.232091,...,681.0,2024-08-17,-2.14,2.10,0,bf16,70.0,llama3_1,135.142,41.887711
9,llama3_1_70b,1154.680726,0.101372,0.000087,42.5,288.958763,68.162163,0.016631,0.109179,0.026642,...,641.0,2024-08-17,-2.58,2.21,0,awq_int4,70.0,llama3_1,135.142,19.244679


In [63]:
df['num_prompts'] = 500
df['sec_per_prompt'] = df['duration'] / df['num_prompts']
df

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,...,CI_lower,CI_upper,guided,quantization,param_size,model_class,input_tok,duration_minutes,num_prompts,sec_per_prompt
0,gpt-4-0314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.00,0.00,0,bf16,NaN,gpt-4-0314,0.000,NaN,500,NaN
1,gpt-4o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.49,1.35,0,bf16,NaN,gpt-4o,0.000,NaN,500,NaN
2,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.47,1.55,0,bf16,NaN,gpt-4o-2024-08-06,0.000,NaN,500,NaN
3,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.48,1.43,1,bf16,NaN,gpt-4o-2024-08-06,0.000,NaN,500,NaN
4,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.92,1.83,0,bf16,NaN,gpt-4o-mini,0.000,NaN,500,NaN
5,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.60,1.59,1,bf16,NaN,gpt-4o-mini,0.000,NaN,500,NaN
6,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-2.29,1.79,0,bf16,405.0,llama3_1,135.142,NaN,500,NaN
7,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.98,1.93,1,bf16,405.0,llama3_1,874.592,NaN,500,NaN
8,llama3_1_70b,2513.262675,0.471343,0.000187,42.5,563.751591,273.079628,0.036198,0.440552,0.232091,...,-2.14,2.10,0,bf16,70.0,llama3_1,135.142,41.887711,500,5.026525
9,llama3_1_70b,1154.680726,0.101372,0.000087,42.5,288.958763,68.162163,0.016631,0.109179,0.026642,...,-2.58,2.21,0,awq_int4,70.0,llama3_1,135.142,19.244679,500,2.309361


In [64]:
df.dtypes

model_name           object
duration            float64
emissions           float64
emissions_rate      float64
cpu_power           float64
gpu_power           float64
ram_power           float64
cpu_energy          float64
gpu_energy          float64
ram_energy          float64
energy_consumed     float64
gpu_count           float64
model_id             object
arena_score         float64
95_conf_minus       float64
95_conf_plus        float64
CI                   object
output_tok          float64
date                 object
CI_lower            float64
CI_upper            float64
guided                int64
quantization         object
param_size          float64
model_class          object
input_tok           float64
duration_minutes    float64
num_prompts           int64
sec_per_prompt      float64
dtype: object

In [65]:
new_column_order = [
    'model_id',
    'model_name',
    'model_class',        
    'param_size', 
    'gpu_count',        
    'quantization',       
    'guided',             
    'num_prompts',        
    'output_tok',         
    'input_tok',          
    'arena_score',
    'CI', 
    'CI_lower',
    'CI_upper',        
    '95_conf_plus',       
    '95_conf_minus',      
    'duration',
    'duration_minutes',
    'sec_per_prompt',
    'cpu_power',
    'gpu_power',
    'ram_power',
    'cpu_energy',
    'gpu_energy',
    'ram_energy',
    'energy_consumed',     
    'emissions',
    'emissions_rate'
]

df = df[new_column_order]

df

,model_id,model_name,model_class,param_size,gpu_count,quantization,guided,num_prompts,output_tok,input_tok,...,sec_per_prompt,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,energy_consumed,emissions,emissions_rate
0,gpt-4-0314,gpt-4-0314,gpt-4-0314,NaN,NaN,bf16,0,500,423.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,gpt-4o,gpt-4o,gpt-4o,NaN,NaN,bf16,0,500,696.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gpt-4o-2024-08-06,gpt-4o-2024-08-06,gpt-4o-2024-08-06,NaN,NaN,bf16,0,500,616.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,gpt-4o-2024-08-06_guided,gpt-4o-2024-08-06,gpt-4o-2024-08-06,NaN,NaN,bf16,1,500,625.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,gpt-4o-mini,gpt-4o-mini,gpt-4o-mini,NaN,NaN,bf16,0,500,631.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,gpt-4o-mini_guided,gpt-4o-mini,gpt-4o-mini,NaN,NaN,bf16,1,500,678.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,llama3_1_405b,llama3_1_405b,llama3_1,405.0,NaN,bf16,0,500,662.0,135.142,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,llama3_1_405b_guided,llama3_1_405b,llama3_1,405.0,NaN,bf16,1,500,649.0,874.592,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,llama3_1_70b-8gpus,llama3_1_70b,llama3_1,70.0,8.0,bf16,0,500,681.0,135.142,...,5.026525,42.5,563.751591,273.079628,0.036198,0.440552,0.232091,0.708841,0.471343,0.000187
9,llama3_1_70b_awq_int4-4gpus,llama3_1_70b,llama3_1,70.0,4.0,awq_int4,0,500,641.0,135.142,...,2.309361,42.5,288.958763,68.162163,0.016631,0.109179,0.026642,0.152451,0.101372,0.000087


In [66]:
# Define the relative path to the target directory
save_path = '../results/data/results.parquet'

# Save the DataFrame to the Parquet file
df.to_parquet(save_path, index=False)